# Dueling Networks implementation 
This is the discrete version of Deep Deterministic Policy Gradient (AKA DQN) but with dueling networks.  
Based on the paper *Dueling Network Architectures for Deep Reinforcement Learning* from Google Deepmind.  

Unlike the paper's implementation, it does not start with a CNN for graphical input, but uses n continuous inputs for physical environments like CartPole.

Implemented by Kevin Cotellesso and Gemini.  


First, we set up stuff we need.


In [1]:
from collections import deque
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import numpy as np
import copy # For copying to target network
import tqdm
import random

In [2]:
env = gym.make('CartPole-v1')
render_env = gym.make('CartPole-v1', render_mode="human")
n_state = int(np.prod(env.observation_space.shape))
n_action = env.action_space.n
print("# of state", n_state)
print("# of action", n_action)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# of state 4
# of action 2
Device: cuda


In [3]:
# Standard function to run an episode of a policy in an env
def run_episode(env, policy, render=False):
    """ Runs an episode and return the total reward """
    obs = env.reset()[0]
    states = []
    rewards = []
    actions = []
    while True:
        if render:
            env.render()

        states.append(obs)
        action = int(policy(obs))
        actions.append(action)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        rewards.append(reward)
        if done:
            break

    return states, actions, rewards


In [4]:
# The ReplayBuffer holds a bunch of trajectories to sample from. Good times
# TODO Understand rank-based prioritized replay a la Schaul et al. 2016
class ReplayBuffer:
    def __init__(self, size, prioritized=False, alpha=0.7, beta=0.5):
        self.size = size                    # The maximum length of the buffer
        self.buff = []                      # The actual replay buffer
        self.priorities = []                # The priority of each item in the buffer
        self.prioritized = prioritized      # Flag for whether to use prioritization of experiences
        self.alpha = alpha                  # Parameter for prioritizing
        self.beta = beta                    # Parameter for prioritizing
        self.position = 0                   # An index that kinda cycles through the entire buffer.
                                            # At any point, it is the position of the oldest value in the buffer. (deque behavior)

    def add(self, obs, act, reward, next_obs, done):
        # Store the maximum priority for TODO
        max_prio = max(self.priorities) if self.buff else 1.0
        
        # The piece of experience we're storing
        transition = [obs, act, reward, next_obs, done]
        
        # If the buffer isn't full yet...
        if len(self.buff) < self.size:
            self.buff.append(transition)                # Add the new experience.
            self.priorities.append(max_prio)            # This experience gets same priority as the highest experience so far (good until proven bad)
        # If the buffer is full...
        else: 
            self.buff[self.position] = transition       # Replace the oldest trajectory
            self.priorities[self.position] = max_prio   # and give it max priority
            
        self.position = (self.position + 1) % self.size # The oldest trajectory is now the one to the right.

    def sample(self, sample_size):

        # If buffer isn't full, sample from the whole buffer
        if len(self.buff) < sample_size:
            sample_size = len(self.buff)
        
        # If prioritization is enabled....
        if self.prioritized:
            # Rank-based prioritization
            # Sort by priorities descending
            sorted_indices = np.argsort(self.priorities)[::-1]
            
            # Since the indices are sorted by priority, they are ranked now. These are the ranks:
            ranks = np.arange(1, len(self.buff) + 1) # e.g. [1,     2,      3,      4,      5]

            # P(i) = (1/rank)^alpha / sum((1/rank)^alpha)
            probs = (1 / ranks) ** self.alpha        # e.g. [1.00,  0.62,   0.46,   0.38,   0.32]
            probs /= probs.sum()                     # e.g. [0.36,  0.22,   0.17,   0.14,   0.12] # Sum up to 1
            # TL;DR: convert ordinal numbers (rank) to probabilities summing to 1.
            # alpha=0: all probabilities are the same
            # alpha->inf: rank 1 approaches 100% probability
            
            # Sample indices based on probabilities
            # choose one of [0,1,2,3,4] based on probabilities [0.36,  0.22,   0.17,   0.14,   0.12]
            sampled_ranks = np.random.choice(len(self.buff), sample_size, p=probs, replace=False)
            # Pick those indices from the list of indices
            indices = sorted_indices[sampled_ranks]
            
            # Importance sampling weights
            weights = (len(self.buff) * probs[sampled_ranks]) ** (-self.beta)
            weights /= weights.max()
            weights = torch.FloatTensor(weights).to(device)
            # Higher weight goes to the lower probabilities

        # If prioritization is disabled...
        else:
            indices = np.random.choice(len(self.buff), sample_size, replace=False)
            weights = torch.ones(sample_size).to(device)
            
        sample = [self.buff[i] for i in indices]
        
        obs = torch.FloatTensor(np.array([exp[0] for exp in sample])).to(device)
        act = torch.LongTensor([exp[1] for exp in sample]).to(device)
        reward = torch.FloatTensor([exp[2] for exp in sample]).to(device)
        next_obs = torch.FloatTensor(np.array([exp[3] for exp in sample])).to(device)
        done = torch.FloatTensor([exp[4] for exp in sample]).to(device)
        
        return obs, act, reward, next_obs, done, indices, weights

    def update_priorities(self, indices, td_errors):
        for i, error in zip(indices, td_errors):
            # More TD error = more priority.
            # (Since high TD error means high capability of learning)
            self.priorities[i] = abs(error) + 1e-5 # add small epsilon to avoid 0 priority

    def __len__(self):
        return len(self.buff)

In [5]:
class DuelingQNet(nn.Module):
    def __init__(self, n_state, n_action):
        super(DuelingQNet, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(n_state, 64),
            nn.ReLU()
        )
        self.value_stream = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, n_action)
        )

    def forward(self, x):
        features = self.fc(x)
        values = self.value_stream(features)
        advantages = self.advantage_stream(features)
        # Combine value and advantage streams into Q-values
        qvals = values + (advantages - advantages.mean(dim=1, keepdim=True))
        return qvals

class Policy():
    def __init__(self, n_state, n_action, eps):
        self.q_net = DuelingQNet(n_state, n_action)
        self.eps = eps
        self.gamma = 0.95

        self.target_net = copy.deepcopy(self.q_net)
        self.target_net.to(device)
        self.target_net.eval()
        self.network_sync_counter = 0
        self.network_sync_freq = 10

        self.optimizer = torch.optim.Adam(self.q_net.parameters(), lr=1e-3)
        self.q_net.to(device)
        # We enable prioritized replay here. Set prioritized=False if you want normal replay.
        self.replaybuff = ReplayBuffer(50000, prioritized=True)

    def update(self, data=None):
        # Update the weights of this policy
        # given a sample of the replay buffer.
        obs, act, reward, next_obs, done, indices, weights = self.replaybuff.sample(32)

        if (self.network_sync_counter == self.network_sync_freq):
            self.target_net.load_state_dict(self.q_net.state_dict())
            self.network_sync_counter = 0
        self.network_sync_counter += 1
        self.optimizer.zero_grad()
        
        with torch.no_grad():
            # DDQN: Select action with active network, evaluate with target network
            next_actions = self.q_net(next_obs).argmax(dim=1)
            next_q_values = self.target_net(next_obs).gather(1, next_actions.unsqueeze(1)).squeeze(1)
            y = reward + self.gamma * (1 - done) * next_q_values

        current_q_values = self.q_net(obs)[range(len(obs)), act]
        
        # Calculate TD errors for priority updates using actual tensor differences
        td_errors = (y - current_q_values).detach().cpu().numpy()
        if self.replaybuff.prioritized:
            self.replaybuff.update_priorities(indices, td_errors)
            
        # Compute the weighted MSE loss
        loss = (weights * F.mse_loss(y, current_q_values, reduction='none')).mean()
        loss.backward()
        self.optimizer.step()
        return loss.item()

    def __call__(self, state):
        if np.random.rand() < self.eps:
            return np.random.choice(n_action)

        if not torch.is_tensor(state):
            state = torch.FloatTensor(state).unsqueeze(0).to(device)
        elif len(state.shape) == 1:
            state = state.unsqueeze(0)
            
        with torch.no_grad():
            Q = self.q_net(state).cpu().numpy()
            act = np.argmax(Q, axis=1)[0]

        return act

Trainin loop

In [6]:
losses_list, reward_list = [], []
policy = Policy(n_state, n_action, 0.5)
update_index = 0
loss = 0
for i in tqdm.tqdm(range(10000)):
    # Each episode:
    # Reset the env
    obs, rew = env.reset()[0], 0
    while True:
        # For each step:

        # Take action in the environment
        act = policy(obs)
        next_obs, reward, terminated, truncated, _ = env.step(act)
        done = terminated or truncated
        rew += reward # Tack on some reward

        update_index += 1

        # If we have enough experience (replay buffer) and it's been a few steps,
        if len(policy.replaybuff) > 2e3 and update_index > 4:
            update_index = 0
            loss = policy.update() # Update the policy (using a sample from replay buffer) 

        # Add this latest step into the replay buffer (experience)
        policy.replaybuff.add(obs, act, reward, next_obs, done)
        obs = next_obs
        if done:
            break # If end of episode. (terminated or truncated)
    
    # If we've gone 500 episodes, set to full exploitation and run a visible episode
    if i > 0 and i % 500 == 0:
        print("itr:({:>5d}) loss:{:>3.4f} reward:{:>3.1f}".format(
            i, np.mean(losses_list[-500:]), np.mean(reward_list[-500:])))
        old_eps = policy.eps
        policy.eps = 0.0
        # run_episode(render_env, policy, render=True)
        policy.eps = old_eps
    policy.eps = max(0.005, policy.eps - 1.0/5000) # Bump epsilon down a little (less exploration)

    losses_list.append(loss), reward_list.append(rew)

  5%|▌         | 501/10000 [00:40<44:21,  3.57it/s] 

itr:(  500) loss:1.0984 reward:61.9


 10%|█         | 1000/10000 [02:08<26:22,  5.69it/s]

itr:( 1000) loss:0.6106 reward:85.3


 15%|█▌        | 1502/10000 [03:36<22:21,  6.33it/s]  

...

 90%|█████████ | 9002/10000 [1:06:09<03:43,  4.46it/s]

itr:( 9000) loss:0.4861 reward:250.1


 95%|█████████▌| 9501/10000 [1:10:52<05:37,  1.48it/s]

itr:( 9500) loss:0.6089 reward:259.9


100%|██████████| 10000/10000 [1:15:21<00:00,  2.21it/s]


In [7]:
policy.eps = 0.0
scores = [sum(run_episode(env, policy, False)[2]) for _ in range(100)]
print("Final score:", np.mean(scores))

import pandas as pd
df = pd.DataFrame({'loss': losses_list, 'reward': reward_list})
df.to_csv("./data/Dueling-target-replay.csv",
          index=False, header=True)


Final score: 114.41
